DEEP LEARNING Sistemas de recomendación basados en contenido

In [35]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import os

folder_path = '/content/drive/My Drive/Recomendadores'  # Ajusta si el nombre es diferente
folder_path = './data/'

os.listdir(folder_path)


['negocios.csv',
 'submissions',
 'test_reviews.csv',
 'train_reviews.csv',
 'usuarios.csv']

In [2]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Función para cargar los datos (asumiendo que tienes los archivos CSV)
def cargar_datos():
    try:
        # Cargar los diferentes conjuntos de datos

        usuarios_df = pd.read_csv(os.path.join(folder_path, 'usuarios.csv'))
        negocios_df = pd.read_csv(os.path.join(folder_path, 'negocios.csv'))
        train_reviews_df = pd.read_csv(os.path.join(folder_path, 'train_reviews.csv'))
        test_reviews_df = pd.read_csv(os.path.join(folder_path, 'test_reviews.csv'))

        #usuarios_df = pd.read_csv('usuarios.csv')
        #negocios_df = pd.read_csv('negocios.csv')
        #train_reviews_df = pd.read_csv('train_reviews.csv')
        #test_reviews_df = pd.read_csv('test_reviews.csv')


        print(f"Datos cargados correctamente:")
        print(f"- Usuarios: {usuarios_df.shape[0]} registros")
        print(f"- Negocios: {negocios_df.shape[0]} registros")
        print(f"- Train reviews: {train_reviews_df.shape[0]} registros")
        print(f"- Test reviews: {test_reviews_df.shape[0]} registros")

        return usuarios_df, negocios_df, train_reviews_df, test_reviews_df

    except FileNotFoundError as e:
        print(f"Error al cargar los archivos: {e}")
        return None, None, None, None

# Función para extraer características de usuarios
def extraer_caracteristicas_usuarios(usuarios_df):
    # Convertir la fecha a características numéricas
    usuarios_df['yelping_since'] = pd.to_datetime(usuarios_df['yelping_since'])
    usuarios_df['años_en_yelp'] = (datetime.now() - usuarios_df['yelping_since']).dt.days / 365

    # Características de interacción social
    usuarios_df['num_amigos'] = usuarios_df['friends'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() else 0)

    # Características de actividad
    usuarios_df['ratio_useful'] = usuarios_df['useful'] / (usuarios_df['review_count'] + 1)
    usuarios_df['ratio_funny'] = usuarios_df['funny'] / (usuarios_df['review_count'] + 1)
    usuarios_df['ratio_cool'] = usuarios_df['cool'] / (usuarios_df['review_count'] + 1)

    # Características de popularidad
    usuarios_df['popularidad'] = usuarios_df['fans'] / (usuarios_df['review_count'] + 1)

    # Suma total de cumplidos recibidos
    usuarios_df['total_compliments'] = (
        usuarios_df['compliment_hot'] + usuarios_df['compliment_more'] +
        usuarios_df['compliment_profile'] + usuarios_df['compliment_cute'] +
        usuarios_df['compliment_list'] + usuarios_df['compliment_note'] +
        usuarios_df['compliment_plain'] + usuarios_df['compliment_cool'] +
        usuarios_df['compliment_funny'] + usuarios_df['compliment_writer'] +
        usuarios_df['compliment_photos']
    )

    return usuarios_df

# Función para extraer características de negocios
def extraer_caracteristicas_negocios(negocios_df):

    # Extraer número de categorías
    negocios_df['num_categorias'] = negocios_df['categories'].apply(
        lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() else 0
    )

    return negocios_df


# Función para unificar los datasets
def unificar_datasets(usuarios_df, negocios_df, train_reviews_df, test_reviews_df):
    # Para el conjunto de entrenamiento
    print("Unificando conjunto de entrenamiento...")
    train_completo = train_reviews_df.copy()

    # Agregar características de usuario
    caracteristicas_usuario = [
        'review_count', 'useful', 'funny', 'cool', 'fans', 'average_stars',
        'años_en_yelp', 'num_amigos', 'total_compliments', 'ratio_useful',
        'ratio_funny', 'ratio_cool', 'popularidad'
    ]

    train_completo = pd.merge(
        train_completo,
        usuarios_df[['user_id'] + caracteristicas_usuario],
        on='user_id',
        how='left',
        suffixes=('', '_usuario')
    )

    # Agregar características del negocio
    caracteristicas_negocio = [
        'stars', 'review_count', 'is_open', 'num_categorias', 'address' , 'city',
        'state', 'postal_code', 'latitude', 'longitude' , 'attributes','is_open',
        'categories', 'hours'
    ]

    train_completo = pd.merge(
        train_completo,
        negocios_df[['business_id'] + caracteristicas_negocio],
        on='business_id',
        how='left',
        suffixes=('', '_negocio')
    )

    # Renombrar columnas para evitar confusión
    train_completo.rename(columns={
        'stars_negocio': 'promedio_estrellas_negocio',
        'stars': 'estrellas_review',
        'review_count_usuario': 'num_reviews_usuario',
        'review_count_negocio': 'num_reviews_negocio'
    }, inplace=True)

    # Para el conjunto de prueba
    print("Unificando conjunto de prueba...")
    test_completo = test_reviews_df.copy()

    # Conseguir user_id y business_id para cada review_id en test
    # Necesitamos esta información de train_reviews o alguna otra fuente
    id_mapping = train_reviews_df[['review_id', 'user_id', 'business_id']]
    test_completo = pd.merge(
        test_completo,
        id_mapping,
        on='review_id',
        how='left',
    )

    test_completo = test_completo.rename(columns={'user_id_x': 'user_id'})
    test_completo = test_completo.rename(columns={'business_id_x': 'business_id'})

    test_completo.drop(['business_id_y', 'user_id_y'], axis=1, inplace=True)


    print(test_completo.head())
    print(usuarios_df.head())
    # Agregar las mismas características que para el conjunto de entrenamiento
    test_completo = pd.merge(
        test_completo,
        usuarios_df[['user_id'] + caracteristicas_usuario],
        on='user_id',
        how='left',
        suffixes=('', '_usuario')
    )
    test_completo = pd.merge(
        test_completo,
        negocios_df[['business_id'] + caracteristicas_negocio],
        on='business_id',
        how='left',
        suffixes=('', '_negocio')
    )

    # Renombrar columnas igual que en entrenamiento
    test_completo.rename(columns={
        'stars_negocio': 'promedio_estrellas_negocio',
        'stars': 'estrellas_review',
        'review_count_usuario': 'num_reviews_usuario',
        'review_count_negocio': 'num_reviews_negocio'
    }, inplace=True)

    return train_completo, test_completo

In [3]:
usuarios_df, negocios_df, train_reviews_df, test_reviews_df = cargar_datos()

C:\Users\Lluis\AppData\Local\Temp\ipykernel_7428\2758696083.py:11: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  usuarios_df = pd.read_csv(os.path.join(folder_path, 'usuarios.csv'))


Datos cargados correctamente:
- Usuarios: 699619 registros
- Negocios: 30069 registros
- Train reviews: 967784 registros
- Test reviews: 414765 registros


In [4]:
# Preprocesar datos
print("Extrayendo características de usuarios...")
usuarios_df = extraer_caracteristicas_usuarios(usuarios_df)

Extrayendo características de usuarios...


In [5]:
print("Extrayendo características de negocios...")
negocios_df = extraer_caracteristicas_negocios(negocios_df)

Extrayendo características de negocios...


In [6]:
# Unificar datasets
train_completo, test_completo = unificar_datasets(
    usuarios_df, negocios_df, train_reviews_df, test_reviews_df
)

Unificando conjunto de entrenamiento...
Unificando conjunto de prueba...
                review_id                 user_id             business_id  \
0  ieYPmCImINjPzTDFmEKBKA  79F9QrQSet-b1yRCIM243Q  sXSUzImYOcRRI3xtG2M85g   
1  QIkJ8fZ4yx_QaHahWWszAA  chuM6TBkFHtTwJ6z96Hj1A  Ipt9ga67vVC_2ob3YmVwNA   
2  seR2KhblYMWg-k9zzN6aYA  hF68a0mpu97u0oaryFYhyg  _RG4IByyBR528CMc7DefJA   
3  BToo00Fi5pfJFA5MI2HM5g  G4yX5Q1tFfwSucFOmiyjdA  xxlbRiWWQkk-6LST3Hd12g   
4  FHJAzi1imodBit3RWK7zQA  Srqi1xb7exdB9uRHxDeEkw  LgGqdFLD7-ca0Z9F_q4Fuw   

   useful  funny  cool                                               text  \
0       1      0     1  Amazing coffee and chill atmosphere. The staff...   
1       4      0     2  I pass by this joint every time I make a run t...   
2       2      0     0  Came here when my kitten got very sick by the ...   
3       2      0     0  So I'll preface by saying we did have an overa...   
4       0      0     0  This place is a joke. Worst bar service ever. ...   

 

In [7]:
# Reducir al 20%
#train_reducido = train_completo.sample(frac=0.01, random_state=42)
#test_reducido = test_completo.sample(frac=0.01, random_state=42)  #20% aleatorio reproducible

In [8]:
from sentence_transformers import SentenceTransformer
import pandas as pd

# Si usas una GPU y PyTorch
model_transformers = SentenceTransformer('paraphrase-MiniLM-L3-v2')
model_transformers = model_transformers.to('cuda')  # Mover a GPU si está disponible


In [9]:
# Función optimizada para convertir JSON a texto estructurado
def json_to_string(json_obj):
    import json

    if not json_obj or pd.isna(json_obj):
        return ""

    if isinstance(json_obj, str):
        try:
            json_obj = json.loads(json_obj)
        except:
            return str(json_obj)

    # Convertir el JSON a una cadena estructurada
    json_str = json.dumps(json_obj, sort_keys=True)
    return json_str

# Función optimizada para generar embeddings en lotes
def generate_embeddings_batch(texts_list):
    batch_size = 128
    all_embeddings = []

    for i in range(0, len(texts_list), batch_size):
        batch_texts = texts_list[i:i+batch_size]
        batch_embeddings = model_transformers.encode(batch_texts)
        all_embeddings.extend(batch_embeddings)

    return all_embeddings

# Aplicar a los campos JSON de forma optimizada
def process_json_fields(df):
    # Preparación para hours
    if 'hours' in df.columns:
        print("Procesando campo 'hours'...")
        df['hours_str'] = df['hours'].apply(json_to_string)
        hours_texts = df['hours_str'].tolist()

        # Generar embeddings en lote
        hours_embeddings = generate_embeddings_batch(hours_texts)
        df['hours_embedding'] = hours_embeddings
        df.drop('hours_str', axis=1, inplace=True)

    # Preparación para attributes
    if 'attributes' in df.columns:
        print("Procesando campo 'attributes'...")
        df['attributes_str'] = df['attributes'].apply(json_to_string)
        attr_texts = df['attributes_str'].tolist()

        # Generar embeddings en lote
        attr_embeddings = generate_embeddings_batch(attr_texts)
        df['attributes_embedding'] = attr_embeddings
        df.drop('attributes_str', axis=1, inplace=True)

    return df

# Función principal para aplicar todas las transformaciones
def aplicar_transformaciones(df):
    df_transformado = df.copy()

    # 1. Crear embeddings para la columna 'text'
    print("Generando embeddings para texto...")
    df_transformado['text'] = df_transformado['text'].fillna('')
    texts = df_transformado['text'].apply(str).tolist()

    # Procesar en lotes
    df_transformado['text_embedding'] = generate_embeddings_batch(texts)

    # 2. Procesar campos JSON
    df_transformado = process_json_fields(df_transformado)

    # 3. Continuar con el resto de transformaciones...

    return df_transformado

# Aplicar a train y test
train_transformado = aplicar_transformaciones(train_completo)
test_transformado = aplicar_transformaciones(test_completo)

Generando embeddings para texto...
Procesando campo 'hours'...
Procesando campo 'attributes'...
Generando embeddings para texto...
Procesando campo 'hours'...
Procesando campo 'attributes'...


In [36]:
test_transformado.head(2)

,review_id,user_id,business_id,useful,funny,cool,text,date,review_count,useful_usuario,...,postal_code,latitude,longitude,attributes,is_open,categories,hours,text_embedding,hours_embedding,attributes_embedding
0,ieYPmCImINjPzTDFmEKBKA,79F9QrQSet-b1yRCIM243Q,sXSUzImYOcRRI3xtG2M85g,1,0,1,Amazing coffee and chill atmosphere. The staff...,2018-01-29 04:33:28,37.0,32.0,...,70112,29.958099,-90.072059,"{'RestaurantsAttire': ""'casual'"", 'Restaurants...",1,"Breakfast & Brunch, Restaurants, Food, Coffee ...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-15:0', '...","[0.15144117, 0.03175736, 0.05149355, -0.139723...","[0.110980906, -0.18648835, -0.10360865, -0.172...","[0.009744417, -0.19596976, -0.3500622, -0.0797..."
1,QIkJ8fZ4yx_QaHahWWszAA,chuM6TBkFHtTwJ6z96Hj1A,Ipt9ga67vVC_2ob3YmVwNA,4,0,2,I pass by this joint every time I make a run t...,2011-01-10 03:10:49,120.0,362.0,...,46226,39.855500,-86.085553,"{'GoodForKids': 'True', 'WiFi': ""u'free'"", 'Am...",1,"Breakfast & Brunch, Greek, American (Tradition...","{'Monday': '7:0-15:0', 'Tuesday': '7:0-15:0', ...","[0.03417016, 0.106515735, 0.09201696, -0.09005...","[0.1437017, -0.16942574, -0.15980779, -0.16582...","[0.15235811, -0.24481484, -0.30631393, -0.0305..."


In [35]:
train_transformado.head(2)

,review_id,user_id,business_id,estrellas_review,useful,funny,cool,text,date,review_count,...,postal_code,latitude,longitude,attributes,is_open,categories,hours,text_embedding,hours_embedding,attributes_embedding
0,ZZO43qKB-s65zplC8RfJqw,-1BSu2dt_rOAqllw9ZDXtA,smkZq4G1AOm4V6p3id5sww,5.0,0,0,0,Fantastic fresh food. The greek salad is amazi...,2016-09-30 15:49:32,9.0,...,33626,28.069486,-82.633828,"{'Caters': 'True', 'NoiseLevel': ""u'average'"",...",1,"Restaurants, Greek","{'Monday': '11:0-21:0', 'Tuesday': '11:0-21:0'...","[-0.23936185, -0.004006734, -0.20950039, -0.27...","[0.06389862, -0.072071284, -0.18840642, -0.203...","[0.10582277, -0.1626439, -0.36713025, -0.02048..."
1,vojXOF_VOgvuKD95gCO8_Q,xpe178ng_gj5X6HgqtOing,96_c_7twb7hYRZ9HHrq01g,1.0,2,0,1,Been a patient at Largo Med/Diagnostic Clinic ...,2020-12-09 14:39:51,31.0,...,33770,27.915015,-82.803637,"{'ByAppointmentOnly': 'True', 'AcceptsInsuranc...",1,"Doctors, Medical Centers, Health & Medical, We...",NaN,"[0.02633458, 0.11938448, 0.037573874, -0.05150...","[-0.021862583, -0.045092806, 0.34932464, -0.16...","[-0.065103695, -0.10021095, -0.30772895, -0.17..."


In [10]:
# train_df = train_reducido
train_df = train_transformado
test_df = test_transformado

In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,LeakyReLU, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
import ast
import os

# Extraer los embeddings como características
X_train_embeddings = np.concatenate(
    [np.stack(train_df[col]) for col in ['text_embedding', 'attributes_embedding', 'hours_embedding']],
    axis=1
)


# Seleccionar características numéricas adicionales
numerical_features = [
    'useful', 'funny', 'cool',              # Características de la reseña
    'review_count', 'useful_usuario', 'funny_usuario', 'cool_usuario',
    'fans', 'average_stars', 'años_en_yelp', 'num_amigos',
    'total_compliments', 'ratio_useful', 'ratio_funny', 'ratio_cool', 'popularidad',  # Usuario
    'num_reviews_negocio', 'is_open', 'num_categorias',  # Negocio
    'latitude', 'longitude'                 # Ubicación
]

# Verificar que las características numéricas existan en ambos conjuntos
valid_features = [feat for feat in numerical_features if feat in train_df.columns and feat in test_df.columns]
print(f"Características numéricas utilizadas: {valid_features}")

# Preparar características numéricas
X_train_numerical = train_df[valid_features].fillna(0).copy()

# Normalizar características numéricas
scaler = StandardScaler()
X_train_numerical_scaled = scaler.fit_transform(X_train_numerical)

# Combinar embeddings con características numéricas
X_train = np.hstack((X_train_embeddings, X_train_numerical_scaled))

# Variable objetivo
y_train = train_df['estrellas_review'].values

# Dividir conjunto de entrenamiento para validación
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)

# Definir la arquitectura de la red neuronal
print("Construyendo el modelo...")
model = Sequential([
    Input(shape=(X_train.shape[1],)),  # input shape como capa inicial

    Dense(512),
    LeakyReLU(alpha=0.01),
    Dropout(0.4),

    Dense(256),
    LeakyReLU(alpha=0.01),
    Dropout(0.3),

    Dense(128),
    LeakyReLU(alpha=0.01),
    Dropout(0.2),

    Dense(64),
    LeakyReLU(alpha=0.01),
    Dropout(0.15),

    Dense(32),
    LeakyReLU(alpha=0.01),
    Dense(1)  # salida regresión
])

# Compilar el modelo
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mae'])

# Definir early stopping para evitar sobreajuste
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Entrenar el modelo
print("Entrenando el modelo...")
history = model.fit(
    X_train_split, y_train_split,
    epochs=30,
    batch_size=256,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)

# Evaluar el modelo en el conjunto de validación
val_loss, val_mae = model.evaluate(X_val, y_val)
print(f"Pérdida en validación: {val_loss:.4f}")
print(f"MAE en validación: {val_mae:.4f}")



Características numéricas utilizadas: ['useful', 'funny', 'cool', 'review_count', 'useful_usuario', 'funny_usuario', 'cool_usuario', 'fans', 'average_stars', 'años_en_yelp', 'num_amigos', 'total_compliments', 'ratio_useful', 'ratio_funny', 'ratio_cool', 'popularidad', 'num_reviews_negocio', 'is_open', 'num_categorias', 'latitude', 'longitude']
Construyendo el modelo...


c:\Users\Lluis\anaconda3\envs\Env\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Entrenando el modelo...
Epoch 1/30
3403/3403 ━━━━━━━━━━━━━━━━━━━━ 39s 11ms/step - loss: 1.1173 - mae: 0.7783 - val_loss: 0.7268 - val_mae: 0.6984
Epoch 2/30
3403/3403 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.6106 - mae: 0.5576 - val_loss: 0.5894 - val_mae: 0.5896
Epoch 3/30
3403/3403 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.5799 - mae: 0.5351 - val_loss: 0.5578 - val_mae: 0.5466
Epoch 4/30
3403/3403 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.5674 - mae: 0.5262 - val_loss: 0.5567 - val_mae: 0.5509
Epoch 5/30
3403/3403 ━━━━━━━━━━━━━━━━━━━━ 35s 10ms/step - loss: 0.5550 - mae: 0.5178 - val_loss: 0.5429 - val_mae: 0.5422
Epoch 6/30
3403/3403 ━━━━━━━━━━━━━━━━━━━━ 37s 11ms/step - loss: 0.5473 - mae: 0.5117 - val_loss: 0.5289 - val_mae: 0.5086
Epoch 7/30
3403/3403 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.5418 - mae: 0.5079 - val_loss: 0.5230 - val_mae: 0.5067
Epoch 8/30
3403/3403 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.5373 - mae: 0.5044 - val_loss: 0.5189 - val_mae: 0.4894


In [12]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()

1164

In [31]:
X_train.shape, X_test.shape

((967784, 1174), (414765, 406))

In [14]:
test_df.head()

,review_id,user_id,business_id,useful,funny,cool,text,date,review_count,useful_usuario,...,postal_code,latitude,longitude,attributes,is_open,categories,hours,text_embedding,hours_embedding,attributes_embedding
0,ieYPmCImINjPzTDFmEKBKA,79F9QrQSet-b1yRCIM243Q,sXSUzImYOcRRI3xtG2M85g,1,0,1,Amazing coffee and chill atmosphere. The staff...,2018-01-29 04:33:28,37.0,32.0,...,70112,29.958099,-90.072059,"{'RestaurantsAttire': ""'casual'"", 'Restaurants...",1,"Breakfast & Brunch, Restaurants, Food, Coffee ...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-15:0', '...","[0.15144117, 0.03175736, 0.05149355, -0.139723...","[0.110980906, -0.18648835, -0.10360865, -0.172...","[0.009744417, -0.19596976, -0.3500622, -0.0797..."
1,QIkJ8fZ4yx_QaHahWWszAA,chuM6TBkFHtTwJ6z96Hj1A,Ipt9ga67vVC_2ob3YmVwNA,4,0,2,I pass by this joint every time I make a run t...,2011-01-10 03:10:49,120.0,362.0,...,46226,39.855500,-86.085553,"{'GoodForKids': 'True', 'WiFi': ""u'free'"", 'Am...",1,"Breakfast & Brunch, Greek, American (Tradition...","{'Monday': '7:0-15:0', 'Tuesday': '7:0-15:0', ...","[0.03417016, 0.106515735, 0.09201696, -0.09005...","[0.1437017, -0.16942574, -0.15980779, -0.16582...","[0.15235811, -0.24481484, -0.30631393, -0.0305..."
2,seR2KhblYMWg-k9zzN6aYA,hF68a0mpu97u0oaryFYhyg,_RG4IByyBR528CMc7DefJA,2,0,0,Came here when my kitten got very sick by the ...,2015-09-06 15:29:02,24.0,28.0,...,19056,40.163890,-74.898869,NaN,1,"Pets, Veterinarians","{'Monday': '0:0-0:0', 'Tuesday': '0:0-0:0', 'W...","[0.15735601, -0.030702492, 0.08792657, -0.1360...","[0.11106503, -0.09983228, -0.0869047, -0.24772...","[-0.021862682, -0.04509291, 0.34932455, -0.168..."
3,BToo00Fi5pfJFA5MI2HM5g,G4yX5Q1tFfwSucFOmiyjdA,xxlbRiWWQkk-6LST3Hd12g,2,0,0,So I'll preface by saying we did have an overa...,2015-09-14 00:49:17,70.0,188.0,...,70124,30.005495,-90.118204,"{'Ambience': ""{'romantic': False, 'intimate': ...",1,"Hot Dogs, American (New), Barbeque, Burgers, C...","{'Monday': '0:0-0:0', 'Tuesday': '11:0-21:0', ...","[0.14244242, 0.026952516, 0.020397794, -0.1715...","[0.08691883, -0.09272007, -0.1432917, -0.17764...","[0.117234066, -0.1537287, -0.34853742, -0.1089..."
4,FHJAzi1imodBit3RWK7zQA,Srqi1xb7exdB9uRHxDeEkw,LgGqdFLD7-ca0Z9F_q4Fuw,0,0,0,This place is a joke. Worst bar service ever. ...,2015-07-24 01:03:40,4.0,2.0,...,33706,27.733374,-82.748305,"{'BusinessAcceptsCreditCards': 'True', 'WiFi':...",1,"Resorts, Hotels, Hotels & Travel, Event Planni...","{'Monday': '0:0-0:0', 'Tuesday': '0:0-0:0', 'W...","[-0.13347706, 0.064473465, -0.060895562, -0.21...","[0.11106503, -0.09983228, -0.0869047, -0.24772...","[-0.05884879, -0.14365351, -0.3195909, 0.09658..."


In [17]:
set(train_df.columns) - set(test_df.columns)

{'promedio_estrellas_negocio'}

In [32]:
model.predict(X_train[0:3])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step


array([[4.8737583],
       [1.224365 ],
       [4.757729 ]], dtype=float32)

In [ ]:
X_test_embeddings = np.concatenate(
    [np.stack(test_df[col]) for col in ['text_embedding', 'attributes_embedding', 'hours_embedding']],
    axis=1
)
X_test_numerical = test_df[valid_features].fillna(0).copy()
X_test_numerical_scaled = scaler.fit_transform(X_test_numerical)

# Combinar embeddings con características numéricas
X_test = np.hstack((X_test_embeddings, X_test_numerical_scaled))

In [41]:
X_train.shape,X_test.shape

((414765, 1174), (414765, 1174))

In [27]:
# Hacer predicciones en el conjunto de prueba
print("Realizando predicciones...")
predictions = model.predict(X_test)

# Asegurarse de que las predicciones estén en el rango correcto (entre 1 y 5)
predictions = np.clip(predictions, 1, 5)

# Crear un DataFrame con los resultados
results_df = pd.DataFrame({
    'review_id': test_df['review_id'],
    'stars': predictions.flatten()
})

# Guardar resultados en CSV
output_file = './data/submissions/predictions_DL2.csv'
results_df.to_csv(output_file, index=False)
print(f"Predicciones guardadas en {output_file}")

# Mostrar las primeras predicciones
print("\nPrimeras 5 predicciones:")
print(results_df.head())

Realizando predicciones...
12962/12962 ━━━━━━━━━━━━━━━━━━━━ 32s 2ms/step
Predicciones guardadas en ./data/submissions/predictions_DL2.csv

Primeras 5 predicciones:
                review_id     stars
0  ieYPmCImINjPzTDFmEKBKA  4.875740
1  QIkJ8fZ4yx_QaHahWWszAA  3.548647
2  seR2KhblYMWg-k9zzN6aYA  3.218649
3  BToo00Fi5pfJFA5MI2HM5g  3.616753
4  FHJAzi1imodBit3RWK7zQA  1.082923


In [ ]:
results_df['stars'] = results_df['stars'].round()
results_df.to_csv('./data/submissions/DL_rounded2.csv', index=False)